### Loading libraries

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait as wait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
from bs4 import BeautifulSoup
from time import sleep
from tqdm import tqdm
import pandas as pd
import os

from sgp4.api import days2mdhms
from sgp4 import exporter
from pprint import pprint
from sgp4.api import jday
from sgp4.api import Satrec

import tkinter as tk
from tkinter import filedialog
from tkinter import messagebox
from tkinter import ttk
from cProfile import label
import easygui
import numpy as np
from collections import defaultdict
import time
from PyQt6.QtWidgets import QApplication, QWidget, QVBoxLayout, QLineEdit, QComboBox, QPushButton, QLabel, QHBoxLayout, QMessageBox
from PyQt6.QtCore import QTimer, QTime, QThread, pyqtSignal, QObject, Qt
import sys
import time


import joblib

### Loading TLE from SpaceTrack

In [4]:
name = easygui.fileopenbox()

In [5]:
service = Service(name)
browser = webdriver.Chrome(service=service)
url = 'https://www.space-track.org/auth/login'
browser.get(url)
id = browser.find_element(By.XPATH,'//*[@id="identity"]')
password = browser.find_element(By.XPATH,'//*[@id="password"]')
id.send_keys('assan256@mail.ru')
password.send_keys('MisterJunior2025!')
login = browser.find_element(By.XPATH,'//*[@id="login-panel-body"]/form/div[3]/input')
login.click()
recent3LE = browser.find_element(By.XPATH, '//*[@id="tab"]/li[8]/a')
recent3LE.click()
TLE = browser.find_element(By.XPATH, '//*[@id="recent"]/div[1]/div[2]/div/div[1]/div/div[1]/ul[1]/li[2]/a')
TLE.click()
browser.switch_to.window(browser.window_handles[1])
TLEParse = browser.find_element(By.XPATH, '/html/body/pre')
objectName = TLEParse.text.split('\n')[::3]
objectName = [x.split('0 ')[1] for x in objectName]
firstLine = TLEParse.text.split('\n')[1::3]
secondLine = TLEParse.text.split('\n')[2::3]

### Sorting Spacecrafts by high

In [7]:
def dictionaries():
    global orbits, jd, fr, position, velocity, timePrediction, r, t, n, rdot, tdot, ndot, covariance
    orbits = {'lowOrbit' : {'objectName' :[], 'firstLine':[], 'secondLine':[]}, 'mediumOrbit' : {'objectName':[], 'firstLine':[], 'secondLine':[]},
             'highOrbit' : {'objectName':[], 'firstLine':[], 'secondLine':[]}} 
    jd = defaultdict(list)
    fr = defaultdict(list)
    position = defaultdict(list)
    velocity = defaultdict(list)
    timePrediction = defaultdict(list)
    covariance = defaultdict(list)
    r = defaultdict(list)
    t = defaultdict(list)
    n = defaultdict(list)
    rdot = defaultdict(list) 
    tdot = defaultdict(list) 
    ndot = defaultdict(list)
    covariance = defaultdict(list)

In [8]:
def highFiltration():
    dictionaries()
    for i in range(len(objectName)):
        satellite = Satrec.twoline2rv(firstLine[i], secondLine[i])
        month, day, hour, minute, second = days2mdhms(satellite.epochyr, satellite.epochdays)
        year = satellite.epochyr + 2000
        jd, fr = jday(year, month, day, hour, minute, second)
        e, r, v = satellite.sgp4(jd, fr)
        if (np.sqrt(r[0]**2 + r[1]**2 + r[2]**2) - 6371) < 2000:
            try:
                orbits['lowOrbit']['objectName'].index(objectName[i])
            except:
                orbits['lowOrbit']['objectName'].append(objectName[i]) 
            else:
                orbits['lowOrbit']['objectName'].append(f'{objectName[i]} {i}')
            orbits['lowOrbit']['firstLine'].append(firstLine[i])
            orbits['lowOrbit']['secondLine'].append(secondLine[i])
        if ((np.sqrt(r[0]**2 + r[1]**2 + r[2]**2) - 6371)>=2000  and   (np.sqrt(r[0]**2 + r[1]**2 + r[2]**2) - 6371) <= 35000):
            try:
                orbits['mediumOrbit']['objectName'].index(objectName[i])
            except:
                orbits['mediumOrbit']['objectName'].append(objectName[i]) 
            else:
                orbits['mediumOrbit']['objectName'].append(f'{objectName[i]} {i}')
            orbits['mediumOrbit']['firstLine'].append(firstLine[i])
            orbits['mediumOrbit']['secondLine'].append(secondLine[i])
        if (np.sqrt(r[0]**2 + r[1]**2 + r[2]**2) - 6371) > 35000:
            try:
                orbits['highOrbit']['objectName'].index(objectName[i])
            except:
                orbits['highOrbit']['objectName'].append(objectName[i]) 
            else:
                orbits['highOrbit']['objectName'].append(f'{objectName[i]} {i}')
            orbits['highOrbit']['firstLine'].append(firstLine[i])
            orbits['highOrbit']['secondLine'].append(secondLine[i])  

In [9]:
def predictionA():
    for orb in ['lowOrbit', 'mediumOrbit', 'highOrbit']:
        if selection in orbits[orb]['objectName']:
            for i in range(len(orbits[orb]['objectName'])):
                satellite = Satrec.twoline2rv(orbits[orb]['firstLine'][i], orbits[orb]['secondLine'][i])
                start_pred = start
                while start_pred <= start + t_pred:
                    timePrediction[orbits[orb]['objectName'][i]].append(start_pred)
                    start_pred += t_step
                timePrediction[orbits[orb]['objectName'][i]] = [pd.to_datetime(x, unit = 's') for x in timePrediction[orbits[orb]['objectName'][i]]]
                jd[orbits[orb]['objectName'][i]] = np.array([jday(x.year, x.month, x.day, x.hour, x.minute, x.second)[0] for x in timePrediction[orbits[orb]['objectName'][i]]])
                fr[orbits[orb]['objectName'][i]] = np.array([jday(x.year, x.month, x.day, x.hour, x.minute, x.second)[1] for x in timePrediction[orbits[orb]['objectName'][i]]])
                e, r, v = satellite.sgp4_array(jd[orbits[orb]['objectName'][i]], fr[orbits[orb]['objectName'][i]])
                position[orbits[orb]['objectName'][i]] = np.array(r)
                velocity[orbits[orb]['objectName'][i]] = np.array(v)

In [10]:
def coov():
    for nama in position.keys():
        for i in range(len(position[nama])):
            R_hat = position[nama][i] / np.linalg.norm(position[nama][i])
            v_radial = np.dot(velocity[nama][i], R_hat) * R_hat
            v_T = velocity[nama][i] - v_radial
            T_hat = v_T / np.linalg.norm(v_T)
            N_hat = np.cross(R_hat, T_hat)
            r[nama].append(np.dot(position[nama][i], R_hat))
            t[nama].append(np.dot(position[nama][i], T_hat))
            n[nama].append(np.dot(position[nama][i], N_hat))
            rdot[nama].append(np.dot(velocity[nama][i], R_hat))
            tdot[nama].append(np.dot(velocity[nama][i], T_hat))
            ndot[nama].append(np.dot(velocity[nama][i], N_hat))
        covariance[nama].append(np.cov(np.array([r[nama], t[nama], n[nama], rdot[nama], tdot[nama], ndot[nama]])))

In [11]:
def collisionA():
    global dangerous1, final, dangerous2 
    final = pd.DataFrame(columns = ['objectName', 'miss_distance', 'relative_speed','relative_position_r','relative_position_t', 'relative_position_n',
                        'relative_velocity_r','relative_velocity_t','relative_velocity_n','t_ct_r','t_cn_r','t_cn_t','t_crdot_r',
                        't_crdot_t','t_crdot_n','t_ctdot_r','t_ctdot_t','t_ctdot_n','t_ctdot_rdot','t_cndot_r','t_cndot_t',
                        't_cndot_n','t_cndot_rdot','t_cndot_tdot','c_ct_r','c_cn_r','c_cn_t','c_crdot_r','c_crdot_t','c_crdot_n',
                        'c_ctdot_r','c_ctdot_t','c_ctdot_n','c_ctdot_rdot','c_cndot_r','c_cndot_t','c_cndot_n','c_cndot_rdot',
                        'c_cndot_tdot','t_position_covariance_det','c_position_covariance_det','t_sigma_r','c_sigma_r','t_sigma_t',
                        'c_sigma_t','t_sigma_n','c_sigma_n','t_sigma_rdot','c_sigma_rdot','t_sigma_tdot','c_sigma_tdot','t_sigma_ndot',
                        'c_sigma_ndot'])
    dangerous1 = defaultdict(list)
    dangerous2 = defaultdict(list)
    for j in position.keys():
        if j != selection:
            for i in range(len(position[j])):
                if len(position[j]) < len(position[selection]):
                    dif = len(position[selection]) - len(position[j])
                    miss_dist = np.sqrt((position[selection][i+dif][0] - position[j][i][0])**2 + (position[selection][i+dif][1] - position[j][i][1])**2 + (position[selection][i+dif][2] - position[j][i][2])**2)
                else:
                    miss_dist = np.sqrt((position[selection][i][0] - position[j][i][0])**2 + (position[selection][i][1] - position[j][i][1])**2 + (position[selection][i][2] - position[j][i][2])**2)
                if  miss_dist <= critical_dictance:
                    relativeSpeed = np.sqrt((velocity[selection][i][0] - velocity[j][i][0])**2 + (velocity[selection][i][1] - velocity[j][i][1])**2 + (velocity[selection][i][2] - velocity[j][i][2])**2)
                    dangerous2['object'].append(j)
                    for orb in ['lowOrbit', 'mediumOrbit', 'highOrbit']:
                        if selection in orbits[orb]['objectName']:
                            dangerous2['id'].append(orbits[orb]['firstLine'][orbits[orb]['objectName'].index(j)][2:8])
                    dangerous2['time'].append(timePrediction[j][i])
                    dangerous2['x'].append(position[j][i][0])
                    dangerous2['y'].append(position[j][i][1])
                    dangerous2['z'].append(position[j][i][2])
                    dangerous2['vx'].append(velocity[j][i][0])
                    dangerous2['vy'].append(velocity[j][i][1])
                    dangerous2['vz'].append(velocity[j][i][2])
                    dangerous2[f'{selection} x'].append(position[selection][i][0])
                    dangerous2[f'{selection} y'].append(position[selection][i][1])
                    dangerous2[f'{selection} z'].append(position[selection][i][2])
                    dangerous2[f'{selection} vx'].append(velocity[selection][i][0])
                    dangerous2[f'{selection} vy'].append(velocity[selection][i][1])
                    dangerous2[f'{selection} vz'].append(velocity[selection][i][2])  
                    dangerous2['miss_distance'].append(miss_dist)

                    R_hat = position[selection][i] / np.linalg.norm(position[selection][i])
                    v_radial = np.dot(velocity[selection][i], R_hat) * R_hat
                    v_T = velocity[selection][i] - v_radial
                    T_hat = v_T / np.linalg.norm(v_T)
                    N_hat = np.cross(R_hat, T_hat)

                    R_hat1 = position[j][i] / np.linalg.norm(position[j][i])
                    v_radial1 = np.dot(velocity[j][i], R_hat1) * R_hat1
                    v_T1 = velocity[j][i] - v_radial1
                    T_hat1 = v_T / np.linalg.norm(v_T1)
                    N_hat1 = np.cross(R_hat, T_hat1)
                    
                    
                    delta_r = position[j][i] - position[selection][i]
                    delta_v = velocity[j][i] - velocity[selection][i]
                    r_R = np.dot(delta_r, R_hat)
                    r_T = np.dot(delta_r, T_hat)
                    r_N = np.dot(delta_r, N_hat)
                    v_R = np.dot(delta_v, R_hat)
                    v_T = np.dot(delta_v, T_hat)
                    v_N = np.dot(delta_v, N_hat)
                    r_distance = np.sqrt(r_R**2 + r_T**2 + r_N**2)
                    r_velocity = np.sqrt(v_R**2 + v_T**2 + v_N**2)
                    dangerous1['object'].append(j)
                    dangerous1['id'] = dangerous2['id']
                    dangerous1['time'].append(timePrediction[j][i])

                    dangerous1['relative_position_r'].append(round(r_R,3))
                    dangerous1['relative_position_t'].append(round(r_T,3))
                    dangerous1['relative_position_n'].append(round(r_N,3))
                    dangerous1['relative_velocity_r'].append(round(v_R,3))
                    dangerous1['relative_velocity_t'].append(round(v_T,3))
                    dangerous1['relative_velocity_n'].append(round(v_N,3))
                    dangerous1['miss_distance'].append(round(np.sqrt(r_R**2 + r_T**2 + r_N**2),3))
                    dangerous1['relative_speed'].append(round(np.sqrt(v_R**2 + v_T**2 + v_N**2),3))
    if len(dangerous1)>0:
    #for key in dangerous1[object]:
        final[final.columns[0]] = [x for x in dangerous1['object']] 
        final[final.columns[1]] = [x for x in dangerous1['miss_distance']] 
        final[final.columns[2]] = [x for x in dangerous1['relative_speed']] 
        final[final.columns[3]] = [x for x in dangerous1['relative_position_r']] 
        final[final.columns[4]] = [x for x in dangerous1['relative_position_t']] 
        final[final.columns[5]] = [x for x in dangerous1['relative_position_n']] 
        final[final.columns[6]] = [x for x in dangerous1['relative_velocity_r']] 
        final[final.columns[7]] = [x for x in dangerous1['relative_velocity_t']] 
        final[final.columns[8]] = [x for x in dangerous1['relative_velocity_n']] 
        #final[final.columns[9]] = [x[1][0] for x in covariance[selection]] 
        final[final.columns[9]] = covariance[selection][0][1][0]
        final[final.columns[10]] = covariance[selection][0][2][0]
        final[final.columns[11]] = covariance[selection][0][2][1] 
        final[final.columns[12]] = covariance[selection][0][0][3]
        final[final.columns[13]] = covariance[selection][0][0][4] 
        final[final.columns[14]] = covariance[selection][0][0][5]
        #
        final[final.columns[15]] = covariance[selection][0][2][3] 
        final[final.columns[16]] = covariance[selection][0][2][4]
        final[final.columns[17]] = covariance[selection][0][2][5] 
        #
        final[final.columns[18]] = covariance[selection][0][3][1]
        final[final.columns[19]] = covariance[selection][0][5][0] 
        final[final.columns[20]] = covariance[selection][0][5][1]
        #
        final[final.columns[21]] = covariance[selection][0][5][2] 
        final[final.columns[22]] = covariance[selection][0][5][3]
        final[final.columns[23]] = covariance[selection][0][5][4] 
        #
        final[final.columns[24]] = [x[1][0] for y in dangerous1['object'] for x in covariance[y]] 
        final[final.columns[25]] = [x[2][0] for y in dangerous1['object'] for x in covariance[y]]
        final[final.columns[26]] = [x[2][1] for y in dangerous1['object'] for x in covariance[y]] 
        final[final.columns[27]] = [x[0][3] for y in dangerous1['object'] for x in covariance[y]]
        final[final.columns[28]] = [x[0][4] for y in dangerous1['object'] for x in covariance[y]] 
        final[final.columns[29]] = [x[0][5] for y in dangerous1['object'] for x in covariance[y]]
        #
        final[final.columns[30]] = [x[2][3] for y in dangerous1['object'] for x in covariance[y]] 
        final[final.columns[31]] = [x[2][4] for y in dangerous1['object'] for x in covariance[y]]
        final[final.columns[32]] = [x[2][5] for y in dangerous1['object'] for x in covariance[y]] 
        #
        final[final.columns[33]] = [x[3][1] for y in dangerous1['object'] for x in covariance[y]]
        final[final.columns[34]] = [x[5][0] for y in dangerous1['object'] for x in covariance[y]] 
        final[final.columns[35]] = [x[5][1] for y in dangerous1['object'] for x in covariance[y]]
        #
        final[final.columns[36]] = [x[5][2] for y in dangerous1['object'] for x in covariance[y]] 
        final[final.columns[37]] = [x[5][3] for y in dangerous1['object'] for x in covariance[y]]
        final[final.columns[38]] = [x[5][4] for y in dangerous1['object'] for x in covariance[y]] 
        #
        final[final.columns[39]] = np.linalg.det(covariance[selection])[0]
        final[final.columns[40]] = [np.linalg.det(covariance[y])[0] for y in dangerous1['object']]
        #
        final[final.columns[41]] = covariance[selection][0][0][0]
        final[final.columns[42]] = [x[0][0] for y in dangerous1['object'] for x in covariance[y]]
        final[final.columns[43]] = covariance[selection][0][1][1] 
        #
        final[final.columns[44]] = [x[1][1] for y in dangerous1['object'] for x in covariance[y]]
        final[final.columns[45]] = covariance[selection][0][2][2] 
        final[final.columns[46]] = [x[2][2] for y in dangerous1['object'] for x in covariance[y]]
        #
        final[final.columns[47]] = covariance[selection][0][3][3] 
        final[final.columns[48]] = [x[3][3] for y in dangerous1['object'] for x in covariance[y]]
        final[final.columns[49]] = covariance[selection][0][4][4] 
        #
        final[final.columns[50]] = [x[4][4] for y in dangerous1['object'] for x in covariance[y]]
        final[final.columns[51]] = covariance[selection][0][5][5] 
        final[final.columns[52]] = [x[5][5] for y in dangerous1['object'] for x in covariance[y]]
    dangerous2['risk'] = list(joblib.load('RegRF').predict(final.drop(columns = ['objectName'], axis = 1).values))
    dangerous1['risk'] = dangerous2['risk']
    dangerous2['dangerous']= list(joblib.load('DTree').predict(final.drop(columns = ['objectName'], axis = 1).values))        
    dangerous1['dangerous'] = dangerous2['dangerous']

In [12]:
def showing():
    if len(dangerous1)>0:
        saveDirectory = tk.filedialog.askdirectory()

        with pd.ExcelWriter(f'{saveDirectory}/collision.xlsx') as writer:
            pd.DataFrame(dangerous1).to_excel(writer, sheet_name=f'{selection} RTN')
            pd.DataFrame(dangerous2).to_excel(writer, sheet_name=f'{selection} XYZ')
            final.to_excel(writer, sheet_name=f'{selection} covariance')
    else: 
        def show_no_collisions_dialog():
            """Функция для отображения диалогового окна."""
            msg = QMessageBox()
            msg.setWindowTitle("Результат прогноза")
            msg.setText("Сближений нет")
            msg.setIcon(QMessageBox.Icon.Information)
            msg.exec()
        show_no_collisions_dialog()

In [13]:
def prediction():
    highFiltration()
    predictionA()
    coov()

In [14]:
def collision():
    collisionA()
    showing()

In [15]:
global t_step, t_pred, critical_dictance, start
t_step = 0
t_pred = 0
critical_dictance = 0
start = 0


class SpaceCollision(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Space Collision")
        self.setGeometry(100, 100, 400, 400)

        self.object_list = objectName  # Замените на актуальные данные
        self.selection = ""

        layout = QVBoxLayout()

        # Поиск
        self.search_box = QLineEdit(self)
        self.search_box.setPlaceholderText("Поиск...")
        self.search_box.textChanged.connect(self.update_list)
        layout.addWidget(self.search_box)

        # Список объектов
        self.combobox = QComboBox(self)
        self.combobox.addItems(self.object_list)
        self.combobox.currentTextChanged.connect(self.selected)
        layout.addWidget(self.combobox)

        # Блок ввода start
        start_layout = QHBoxLayout()
        self.start_input = QLineEdit(self)
        self.start_input.setPlaceholderText("Дата начала (ГГГГ.ММ.ДД чч:мм:сс)")
        self.start_input.textChanged.connect(self.set_start)
        start_layout.addWidget(self.start_input)
        layout.addLayout(start_layout)

        # Блок ввода t_step
        t_step_layout = QHBoxLayout()
        self.t_step_input = QLineEdit(self)
        self.t_step_input.setPlaceholderText("Время шага прогноза")
        self.t_step_input.textChanged.connect(self.set_t_step)
        self.t_step_unit = QComboBox(self)
        self.t_step_unit.addItems(["Секунды", "Минуты", "Часы"])
        self.t_step_unit.currentIndexChanged.connect(self.set_t_step)
        t_step_layout.addWidget(self.t_step_input)
        t_step_layout.addWidget(self.t_step_unit)
        layout.addLayout(t_step_layout)

        # Блок ввода t_pred
        t_pred_layout = QHBoxLayout()
        self.t_pred_input = QLineEdit(self)
        self.t_pred_input.setPlaceholderText("Длительность прогноза")
        self.t_pred_input.textChanged.connect(self.set_t_pred)
        self.t_pred_unit = QComboBox(self)
        self.t_pred_unit.addItems(["Секунда", "Минута", "Час", "День", "Неделя"])
        self.t_pred_unit.currentIndexChanged.connect(self.set_t_pred)
        t_pred_layout.addWidget(self.t_pred_input)
        t_pred_layout.addWidget(self.t_pred_unit)
        layout.addLayout(t_pred_layout)

        # Блок ввода критического расстояния
        crit_layout = QHBoxLayout()
        self.critical_dictance_input = QLineEdit(self)
        self.critical_dictance_input.setPlaceholderText("Критическое расстояние (км)")
        self.critical_dictance_input.textChanged.connect(self.set_critical_dictance)
        crit_layout.addWidget(self.critical_dictance_input)
        layout.addLayout(crit_layout)

        # Кнопки
        self.pred_button = QPushButton("Прогноз движения")
        self.pred_button.clicked.connect(self.run_prediction)
        layout.addWidget(self.pred_button)

        self.collision_button = QPushButton("Прогноз столкновения")
        self.collision_button.clicked.connect(self.run_collision)
        layout.addWidget(self.collision_button)

        # Метка вывода текущих значений
        self.output_label = QLabel("")
        self.output_label.setWordWrap(True)
        layout.addWidget(self.output_label)

        self.setLayout(layout)

    def set_start(self):
        global start
        try:
            start = self.start_input.text()
            start = pd.to_datetime(start).timestamp()
            self.output_label.setText(f"Дата начала прогноза: {start} ({int(start)})")
        except Exception:
            self.output_label.setText("Неверный формат даты!")

    def set_t_step(self):
        global t_step
        value = self.t_step_input.text()
        if not value.isdigit():
            return
        value = int(value)
        unit = self.t_step_unit.currentText()
        if unit == "Минуты":
            value *= 60
        elif unit == "Часы":
            value *= 3600
        t_step = value
        self.output_label.setText(f"Время шага прогноза: {t_step} сек")

    def set_t_pred(self):
        global t_pred
        value = self.t_pred_input.text()
        if not value.isdigit():
            return
        value = int(value)
        unit = self.t_pred_unit.currentText()
        if unit == "Секунда":
            value *= 1
        if unit == "Минута":
            value *= 60
        elif unit == "Час":
            value *= 3600
        elif unit == "День":
            value *= 86400
        elif unit == "Неделя":
            value *= 7 * 86400
        t_pred = value
        self.output_label.setText(f"Длительность прогноза: {t_pred} сек")

    def set_critical_dictance(self):
        global critical_dictance
        try:
            critical_dictance = float(self.critical_dictance_input.text())
            self.output_label.setText(f"Критическое расстояние: {critical_dictance} км")
        except ValueError:
            self.output_label.setText("Некорректное значение критического расстояния")

    def selected(self, text):
        global selection
        selection = text
        self.selection = text
        self.output_label.setText(f"Выбран объект: {text}")

    def update_list(self, text):
        filtered = [item for item in self.object_list if text.lower() in item.lower()]
        self.combobox.clear()
        self.combobox.addItems(filtered if filtered else self.object_list)

    def run_prediction(self):
        QApplication.setOverrideCursor(Qt.CursorShape.WaitCursor)
        try:
            prediction()  # твоя функция
        finally:
            QApplication.restoreOverrideCursor()

    def run_collision(self):
        QApplication.setOverrideCursor(Qt.CursorShape.WaitCursor)
        try:
            collision()  # твоя функция
        finally:
            QApplication.restoreOverrideCursor()


if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = SpaceCollision()
    window.show()
    app.exec()